In [3]:
import requests
import json
import pandas as pd
import io

# --- CONFIGURATION ---
# Remplace par les URLs réelles de tes Spaces Hugging Face
MOCK_API_URL = "https://semarmehdi-ibmattritionapi.hf.space"
MODEL_API_URL = "https://atomik31-model-serv-api.hf.space"


def run_test():
    print("1. 📡 Récupération d'un employé depuis la Mock API...")
    try:
        # Appel à la Mock API
        response_mock = requests.get(f"{MOCK_API_URL}/current-employee")
        response_mock.raise_for_status()

        # Récupération du dictionnaire (grâce à ton fix json.loads dans l'API)
        data_split = response_mock.json()

        # Sécurité : Si l'API renvoie encore une string, on la transforme en dict
        if isinstance(data_split, str):
            data_split = json.loads(data_split)

        # RECONSTRUCTION DIRECTE (Plus robuste que read_json)
        # On utilise les clés 'data', 'columns' et 'index' du format "split"
        df_employee = pd.DataFrame(
            data=data_split["data"],
            columns=data_split["columns"],
            index=data_split["index"],
        )

        # Nettoyage de la colonne de métadonnée
        if "current_time" in df_employee.columns:
            df_employee = df_employee.drop(columns=["current_time"])

        print("✅ Donnée récupérée et convertie en DataFrame avec succès.")
        print(
            f"Aperçu des features : {df_employee.columns.tolist()[:5]}... (total {len(df_employee.columns)} cols)"
        )

    except Exception as e:
        print(f"❌ Erreur Étape 1 (Mock API) : {e}")
        return

    print("\n2. 🧠 Envoi des données à l'API de Prédiction...")
    try:
        # On transforme la première ligne du DataFrame en dictionnaire pour le modèle
        # Utiliser to_json() puis json.loads() garantit la compatibilité des types (int64 -> int)
        payload = json.loads(df_employee.iloc[0].to_json())

        # Appel au endpoint /predict
        # Timeout de 120s car le modèle MLflow peut mettre du temps à charger (Cold Start)
        response_pred = requests.post(
            f"{MODEL_API_URL}/predict", json=payload, timeout=120
        )

        if response_pred.status_code == 200:
            result = response_pred.json()
            prediction = result["prediction"]
            status = (
                "🚨 Attrition (Départ probable)"
                if prediction == 1
                else "✅ Loyal (Reste dans l'entreprise)"
            )
            print(f"🚀 Résultat du modèle : {prediction} -> {status}")
        else:
            print(f"⚠️ Erreur de prédiction (Code {response_pred.status_code})")
            print(f"Détail : {response_pred.text}")

    except requests.exceptions.Timeout:
        print(
            "❌ Timeout : L'API de modèle est trop lente à répondre (chargement en cours ?)."
        )
    except Exception as e:
        print(f"❌ Erreur Étape 2 (Model API) : {e}")


if __name__ == "__main__":
    run_test()

1. 📡 Récupération d'un employé depuis la Mock API...
✅ Donnée récupérée et convertie en DataFrame avec succès.
Aperçu des features : ['Age', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome']... (total 34 cols)

2. 🧠 Envoi des données à l'API de Prédiction...
🚀 Résultat du modèle : 0 -> ✅ Loyal (Reste dans l'entreprise)
